# 🛍️ 範例 06：零售會員—顧客消費價值分群 (非監督集群篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「零售電商會員營運、CRM 行銷企劃與顧客關係經理（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以會員營運白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、會員 RFM 資料集介紹 (`customers_rfm.csv`)

本數據集記錄了零售電商 2,000 位會員過去一年的消費交易行為特徵：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 會員營運實務意涵 |
| :--- | :--- | :--- | :--- |
| **顧客代號** | 類別型 (`object`) | CUST_10001 | 單一會員編號（不參與距離運算） |
| **最近一次消費天數_Recency** | 數值型 (`int64`) | 14 (天) | 距今最近一次購買天數（活躍度） |
| **消費頻率次數_Frequency** | 數值型 (`int64`) | 18 (次) | 過去一年累積購買次數（忠誠度） |
| **累計消費金額_Monetary_TWD** | 數值型 (`int64`) | 38,000 (元) | 累積貢獻營收總額（價值度） |
| **平均客單價_TWD** | 數值型 (`int64`) | 2,110 (元) | 單次購買之客單價水準 |
| **主力消費品類** | 類別型 (`object`) | 美妝保養、生鮮食品、母嬰用品、服飾配件、生活小家電 | 核心偏好品類 |
| **會員等級** | 類別型 (`object`) | 鑽石VIP、白金會員、黃金會員、一般會員 | 企業原本人工劃分之會員門檻 |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視 RFM 三維度分佈與關聯性。
3. **特徵相關性與視覺化**：繪製消費天數 (R) vs 累積金額 (M) 散佈圖。
4. **特徵標準化 (StandardScaler)**：消除金額幾萬元與天數幾十天之尺度量綱差異。
5. **手肘法與輪廓係數**：評估 $K=2$ 到 $K=8$ 的分群緊湊度與最佳群數。
6. **K-Means 分群落地與行銷應用**：產生 4 大群像並實作新會員即時分群推論。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入會員 RFM 分群分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並確保中文字體能在散佈圖與集群分佈圖中正常顯示。

---
💡 **AI 提問小抄：**

1. 📌 **非監督式學習的核心精神**
   > 「在先前的範例中我們都有一個 $y$ 標籤（如是否成交、營收），但本範例沒有 $y$。演算法是怎麼在沒有老師告訴它標準答案的情況下，自己把顧客分組的？」


---  
### 【步驟 1】：載入資料與 RFM 結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 customers_rfm.csv 並檢視資料概況
df = pd.read_csv("customers_rfm.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- RFM 數值統計摘要 ---")
display(df[["最近一次消費天數_Recency", "消費頻率次數_Frequency", "累計消費金額_Monetary_TWD"]].describe().round(1))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  檢查 2,000 位會員的 RFM 特徵分佈，觀察天數、次數與金額的四分位距。

---
💡 **AI 提問小抄：**

1. 📌 **RFM 模型的業務重要性**
   > 「請向電商老闆用大白話解釋：為什麼 R（最近消費天數）、F（消費頻次）、M（累計金額）這三個指標，被譽為衡量會員終身價值 (LTV) 最經典的三大黃金指標？」


---  
### 【步驟 2】：特徵相關性與散佈圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製最近天數 vs 累積金額散佈圖
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df,
    x="最近一次消費天數_Recency",
    y="累計消費金額_Monetary_TWD",
    hue="消費頻率次數_Frequency",
    palette="viridis",
    alpha=0.7
)
plt.title("顧客 RFM 分佈散佈圖 (顏色深淺代表消費頻率)")
plt.xlabel("最近一次消費天數 (天，越小代表越活躍)")
plt.ylabel("累計消費金額 (TWD)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  以天數為 X 軸、金額為 Y 軸、頻次為顏色，直觀呈現顧客在二維平面上的聚類分佈。

---
💡 **AI 提問小抄：**

1. 📌 **右上角與左上角象限解讀**
   > 「在圖表左上角（天數很小、金額超高）與右下角（天數很大、金額極低）分別代表什麼樣的顧客族群？行銷資源應該偏重維繫哪一角？」


---  
### 【步驟 3】：特徵標準化 (StandardScaler)

In [ ]:
# [步驟 3 程式碼] 對 RFM 三大數值特徵進行標準化縮放
from sklearn.preprocessing import StandardScaler

rfm_cols = ["最近一次消費天數_Recency", "消費頻率次數_Frequency", "累計消費金額_Monetary_TWD"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[rfm_cols])

print("標準化後的前 3 筆數據矩陣：\n", np.round(X_scaled[:3], 3))
print(f"\n✅ 標準化完成！特徵平均值接近：{X_scaled.mean(axis=0).round(2)}，標準差為：{X_scaled.std(axis=0).round(2)}")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  使用 `StandardScaler` 將金額（幾萬）與次數（個位數）轉化為平均為 0、標準差為 1 的無量綱數值，防止幾何距離被大數字霸凌。

---
💡 **AI 提問小抄：**

1. 📌 **標準化對 K-Means 的決定性影響**
   > 「如果我們省略這一步，直接拿原始金額與天數去跑 K-Means，為什麼演算法會幾乎 100% 只根據『金額』來分組，而徹底無視『消費天數』與『頻率』？」


---  
### 【步驟 4】：手肘法 (Elbow Method) 與輪廓係數決定最佳群數 K

In [ ]:
# [步驟 4 程式碼] 測試 K=2 至 K=7 的慣性 (Inertia) 與手肘圖
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 8)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(k_range, inertias, "bo-", lw=2)
axes[0].set_title("手肘法 (Elbow Method) - 尋找斜率轉折點")
axes[0].set_xlabel("群數 K")
axes[0].set_ylabel("群內平方和 (Inertia)")
axes[0].grid(True)

axes[1].plot(k_range, sil_scores, "rs-", lw=2)
axes[1].set_title("輪廓係數 (Silhouette Score) - 評估群聚分離度")
axes[1].set_xlabel("群數 K")
axes[1].set_ylabel("平均輪廓係數")
axes[1].grid(True)

plt.tight_layout()
plt.show()

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  繪製手肘圖與輪廓係數圖，尋找下降幅度開始變緩的轉折點（通常在 K=4），客觀敲定最佳分群數。

---
💡 **AI 提問小抄：**

1. 📌 **指標挑選權衡**
   > 「在商業實務中，如果輪廓係數顯示 K=2 最高，但行銷部門希望能細分出『VIP、潛力、流失、新客』4 種維度，資料科學家該如何平衡數學指標與商業可行性？」


---  
### 【步驟 5】：K-Means 分群訓練與 4 大顧客群像解讀

In [ ]:
# [步驟 5 程式碼] 執行 K=4 分群並分析各群集特徵均值
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["群集代號"] = kmeans.fit_predict(X_scaled)

print("=== 各群集在原始 RFM 指標上的平均表現 ===")
cluster_summary = df.groupby("群集代號")[rfm_cols].mean().round(1)
cluster_summary["人數規模"] = df["群集代號"].value_counts()
display(cluster_summary)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="最近一次消費天數_Recency", y="累計消費金額_Monetary_TWD", hue="群集代號", palette="tab10", alpha=0.8)
plt.title("K-Means 最終 4 大顧客群聚分佈")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將每位顧客打上群集標籤，計算各群在天數、頻次、累積金額的平均值，並繪製彩色群集分佈圖。

---
💡 **AI 提問小抄：**

1. 📌 **群集人設立體化**
   > 「觀察上方 `cluster_summary` 表格，請幫我將第 0、1、2、3 群分別命名為生動的顧客人設（例如『剛進門的探路客』、『即將說再見的過客』），並為每一群寫出 20 字的行銷行動方針！」


---  
### 【步驟 6】：新會員即時分群歸類 (Inference) 與客製行銷策略

In [ ]:
# [步驟 6 程式碼] 保存模型與前處理器，示範新會員即時分群歸類
import joblib

model_bundle = {"scaler": scaler, "kmeans": kmeans, "features": rfm_cols}
joblib.dump(model_bundle, "model.joblib")
print("✅ 分群模型與標準化縮放器已打包存至 model.joblib！")

# 模擬一位剛結帳的新會員
new_cust = pd.DataFrame([{
    "最近一次消費天數_Recency": 10,
    "消費頻率次數_Frequency": 15,
    "累計消費金額_Monetary_TWD": 42000
}])

loaded = joblib.load("model.joblib")
new_scaled = loaded["scaler"].transform(new_cust)
assigned_cluster = loaded["kmeans"].predict(new_scaled)[0]

print(f"\n🔔 會員自動分群判定完成：該會員歸屬於【群集 {assigned_cluster}】")
print("🎯 觸發自動化行銷：符合高價值 VIP 條件，系統自動發送 LINE 專屬 VIP 私人管家邀請函！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將標準化縮放器與 K-Means 演算法一起封裝存檔，展示如何在生產環境中接收新顧客 RFM 數字並即時推論分群。

---
💡 **AI 提問小抄：**

1. 📌 **即時推論與批次分群**
   > 「在真實電商 CDP（顧客數據平台）中，分群通常是『每天半夜排程全量重新分群』，還是『結帳瞬間即時歸類』？這兩種架構在運算資源上有何考量？」
